# Week 4 Protocol: The Test Run (Pre-Registered)

<a href="https://colab.research.google.com/github/jorshimayor/AI-Engineering-Flow_Research/blob/main/jupyter/week4_test_run_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This runs the frozen protocol, [`protocol/week4_eval_protocol_v1.3.md`](../protocol/week4_eval_protocol_v1.3.md): **480 held-out test problems** per model, three modes (direct, Act-only, ReAct), greedy decoding, each model at the difficulty level fixed by dev calibration (1.5B: level 2, 3B: level 3). The settings come from [`protocol/week4_frozen_config.json`](../protocol/week4_frozen_config.json), and the analysis is the pre-registered code in [`src/protocol_analysis.py`](../src/protocol_analysis.py). Nothing here should be edited after you start.

**Time:** about 7.5 GPU hours in total (about 3.2 h for 1.5B and 4.3 h for 3B, measured during calibration). That's more than one free Colab session, so:
- Set `RUN_MODELS` in step 5 to one model per session, or leave both and just re-run after a disconnect. Finished problems are always skipped.
- Keep **Google Drive on** (step 3), or progress is lost when the runtime resets.
- Kaggle's weekly GPU quota (about 30 h) can do it in one go: import this notebook, then `Settings → Accelerator → GPU T4` and `Internet → On` (and set `USE_DRIVE = False`).

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU, then run again."
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

## 2. Get the code (always the latest)

In [ ]:
REPO = "https://github.com/jorshimayor/AI-Engineering-Flow_Research.git"
BRANCH = "main"
import os
BASE = "/content" if os.path.isdir("/content") else os.getcwd()     # Colab: /content, Kaggle: /kaggle/working
CODE = os.path.join(BASE, "AI-Engineering-Flow_Research")
if not os.path.exists(CODE):
    !git clone --depth 1 --branch {BRANCH} {REPO} {CODE}
%cd {CODE}
!git fetch -q --depth 1 origin {BRANCH} && git checkout -q -B {BRANCH} FETCH_HEAD
!pip install -q "transformers==4.49.0" accelerate
!git log --oneline -1

## 3. Keep results on Google Drive

In [ ]:
USE_DRIVE = True
RESULTS = "results/week4_test"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/AI-Engineering/week4_test"
    os.makedirs(target, exist_ok=True); os.makedirs("results", exist_ok=True)
    if not os.path.islink(RESULTS):
        if os.path.isdir(RESULTS):
            os.rmdir(RESULTS)
        os.symlink(target, RESULTS)
os.makedirs(RESULTS, exist_ok=True)
print("results go to:", os.path.realpath(RESULTS))

## 4. Load the frozen protocol and record what's being run

Records the protocol's commit and SHA-256 fingerprint next to the results, so anyone can check the analysis followed the version committed before the run. Test questions are generated from the fixed seed and checked automatically, but never printed.

In [ ]:
import sys, json, hashlib, subprocess, importlib, datetime
from pathlib import Path
sys.path.insert(0, "src")
subprocess.run(["git", "fetch", "-q", "--unshallow"], capture_output=True)   # full history, to find the freeze commit
import react_tasks_v2; importlib.reload(react_tasks_v2)
from react_tasks_v2 import make_problems, check_problems, leakage_check, GENERATOR_VERSION
from protocol_analysis import load_config

CFG = load_config("protocol/week4_frozen_config.json")
assert GENERATOR_VERSION == CFG["generator_version"], f"generator {GENERATOR_VERSION} != frozen {CFG['generator_version']}: Runtime → Disconnect and delete runtime"
proto = Path(CFG["protocol"])
manifest = dict(
    protocol=str(proto), protocol_version=CFG["protocol_version"],
    protocol_sha256=hashlib.sha256(proto.read_bytes()).hexdigest(),
    protocol_commit=subprocess.run(["git", "log", "-1", "--format=%H %cI", "--", str(proto)], capture_output=True, text=True).stdout.strip(),
    code_commit=subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
    generator_version=GENERATOR_VERSION, gpu=torch.cuda.get_device_name(0))
mf = Path(RESULTS) / "run_manifest.json"
if mf.exists():
    first = json.loads(mf.read_text())
    assert first["protocol_sha256"] == manifest["protocol_sha256"], "The protocol changed since this run started. Stop and investigate."
    print("resuming run started", first["started"])
else:
    manifest["started"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    mf.write_text(json.dumps(manifest, indent=1))
print(json.dumps(manifest, indent=1))

for m, spec in CFG["models"].items():
    c = check_problems(make_problems("test", spec["level"], CFG["n_per_template"]))
    print(f"{m}: level {spec['level']}, {c['n']} test problems, errors {len(c['errors'])}")
    assert not c["errors"] and c["n"] == CFG["n_problems_per_model"]
assert not leakage_check()
print("leakage: none")

## 5. Run

Resumable: re-run this cell after any disconnect. To split the work across sessions, set `RUN_MODELS` to one model at a time.

In [ ]:
import gc
from react_agent import LocalLLM
from run_react_bench import run

RUN_MODELS = list(CFG["models"])          # e.g. ["Qwen/Qwen2.5-1.5B-Instruct"] for one session, then the 3B one

for model in RUN_MODELS:
    level = CFG["models"][model]["level"]
    out = Path(RESULTS) / f"{model.split('/')[-1]}-L{level}.jsonl"
    problems = make_problems("test", level, CFG["n_per_template"])
    llm = LocalLLM(model)
    run(model, CFG["modes"], problems, out, CFG["max_steps"], llm=llm)
    del llm; gc.collect(); torch.cuda.empty_cache()

## 6. Pre-registered analysis

Runs only when both models have all 480 problems in all three modes. Otherwise it reports progress.

In [ ]:
from protocol_analysis import analyse, print_report
rep = analyse(RESULTS, CFG)
if rep["status"] == "complete":
    print_report(rep)
    json.dump(rep, open(Path(RESULTS) / "protocol_report.json", "w"), indent=1, default=str)
else:
    print(rep["status"])
    for m, v in rep["models"].items():
        print(f"  {m}: {v['n_paired']} of {CFG['n_problems_per_model']} problems have both Act and ReAct results")

## 7. Cost and accuracy (Pareto plot)

In [ ]:
import matplotlib.pyplot as plt
if rep["status"] == "complete":
    fig, ax = plt.subplots(figsize=(8, 4.5))
    colors = {"direct": "#BBBBBB", "act": "#54A24B", "react": "#4C78A8"}
    for m, r in rep["models"].items():
        for mode, a in r["accuracy"].items():
            ax.errorbar(a["tokens"], a["acc"] * 100, yerr=[[(a["acc"] - a["ci"][0]) * 100], [(a["ci"][1] - a["acc"]) * 100]],
                        fmt="o" if "1.5B" in m else "s", color=colors[mode], capsize=3)
            ax.annotate(f"{mode} {m.split('-')[1]}", (a["tokens"], a["acc"] * 100), textcoords="offset points", xytext=(6, -3), fontsize=8)
    ax.set_xlabel("tokens per problem (cost)"); ax.set_ylabel("accuracy (%)"); ax.set_title("Test set: cost vs accuracy (95% Wilson CIs)")
    plt.tight_layout(); plt.savefig(Path(RESULTS) / "test_pareto.png", dpi=130); plt.show()

## 8. Save the results

In [ ]:
zip_path = os.path.join(BASE, "week4_test_results.zip")
!cd results && zip -qr {zip_path} week4_test/
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not on Colab: download", zip_path, "from the output panel")